# PyRAT Water Delivery Demo

Demonstrates fetching subjects from the PyRAT API and parsing water delivery records from their comments using `parse_water_from_comment`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import yaml

from ucl_open.pyrat import (
    PyRatClient,
    parse_session_end_from_comment,
    parse_session_start_from_comment,
    parse_water_from_comment,
)

users_path = Path("../src/PyRAT_GUI/users.json")
server_path = Path("../src/PyRAT_GUI/pyrat_server.yaml")

with server_path.open() as f:
    URL = yaml.safe_load(f)["url"]

with users_path.open() as f:
    users_data = json.load(f)

CLIENT_TOKEN = users_data["client_token"]

_available = {name: u["token"] for name, u in users_data["users"].items() if u["token"]}
print("Available users:", list(_available.keys()))

USER = "MysteriousStranger"
USER_TOKEN = _available[USER]
print(f"Using user: {USER}, URL: {URL}")

In [ ]:
client = PyRatClient(url=URL, client_token=CLIENT_TOKEN, user_token=USER_TOKEN)

creds = client.verify_credentials()
print(creds)

In [ ]:
subjects = client.fetch_subjects(limit=50)
print(f"Fetched {len(subjects)} subjects")

In [ ]:
# Parse all structured comment types per subject
water_records = []
session_starts = []
session_ends = []
other_comments = []

for subject in subjects:
    for comment in subject.comments:
        if (w := parse_water_from_comment(comment, subject.eartag_or_id)):
            water_records.append({"eartag_or_id": subject.eartag_or_id, "delivery_date": w.delivery_date, "water_ml": w.water_amount_ml})
        elif (s := parse_session_start_from_comment(comment, subject.eartag_or_id)):
            session_starts.append({"eartag_or_id": subject.eartag_or_id, "workflow": s.workflow, "session_start": s.session_start})
        elif (e := parse_session_end_from_comment(comment, subject.eartag_or_id)):
            session_ends.append({"eartag_or_id": subject.eartag_or_id, "workflow": e.workflow, "session_end": e.session_end})
        elif comment.content:
            other_comments.append({"eartag_or_id": subject.eartag_or_id, "created": comment.created, "content": comment.content})

print(f"Water deliveries: {len(water_records)}, Session starts: {len(session_starts)}, Session ends: {len(session_ends)}, Other: {len(other_comments)}")

In [ ]:
# Subject info
df_subjects = pd.DataFrame([{
    "eartag_or_id": s.eartag_or_id,
    "sex": s.sex,
    "weight_g": s.weight,
    "age_weeks": s.age_weeks,
    "strain": s.strain_name_with_id,
    "room": s.room_name,
    "cage": s.cagenumber,
    "responsible": s.responsible_fullname,
} for s in subjects])
display(df_subjects)

In [ ]:
# Session summary per subject
df_starts = pd.DataFrame(session_starts)
df_water = pd.DataFrame(water_records)

if not df_starts.empty:
    summary = df_starts.groupby("eartag_or_id").agg(
        sessions=("session_start", "count"),
        first_session=("session_start", "min"),
        last_session=("session_start", "max"),
    )
    if not df_water.empty:
        summary = summary.join(df_water.groupby("eartag_or_id")["water_ml"].sum().rename("total_water_ml"))
    display(summary)
else:
    print("No session start comments found.")

In [ ]:
# Session starts
if session_starts:
    display(pd.DataFrame(session_starts).sort_values(["eartag_or_id", "session_start"]))
else:
    print("No session start comments found.")

In [ ]:
# Session ends
if session_ends:
    display(pd.DataFrame(session_ends).sort_values(["eartag_or_id", "session_end"]))
else:
    print("No session end comments found.")

In [ ]:
# Other (unstructured) comments
if other_comments:
    display(pd.DataFrame(other_comments).sort_values(["eartag_or_id", "created"]))
else:
    print("No unstructured comments found.")

client.close()